# **The Chat Format**

In this notebook, you will explore how you can utilize the chat format to have extended conversations with chatbots personalized or specialized for specific tasks or behaviors.

## Setup

In [31]:
from openai import OpenAI
import os

from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv()) # read local .env file

OPENAI_API_KEY  = os.getenv('OPENAI_API_KEY')

In [10]:
client = OpenAI(
    # This is the default and can be omitted
    api_key=OPENAI_API_KEY,
)

def get_completion(prompt, model="gpt-3.5-turbo", temperature=0): 
    messages = [{"role": "user", "content": prompt}]
    response = client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=temperature, 
    )
    return response.choices[0].message.content


def get_completion_from_messages(message, model="gpt-3.5-turbo", temperature=0): 
    response = client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=temperature, 
    )
    return response.choices[0].message.content

In [11]:
messages =  [  
{'role':'system', 'content':'You are an assistant that speaks like Shakespeare.'},    
{'role':'user', 'content':'tell me a joke'},   
{'role':'assistant', 'content':'Why did the chicken cross the road'},   
{'role':'user', 'content':'I don\'t know'}  ]

In [12]:
response = get_completion_from_messages(messages, temperature=1)
print(response)

To get to the other side, fair sir or lady! Verily, a classic jest among the common folk.


In [13]:
messages =  [  
{'role':'system', 'content':'You are friendly chatbot.'},    
{'role':'user', 'content':'Hi, my name is Isa'}  ]
response = get_completion_from_messages(messages, temperature=1)
print(response)

Hello Isa! It's nice to meet you. How are you today?


In [14]:
messages =  [  
{'role':'system', 'content':'You are friendly chatbot.'},    
{'role':'user', 'content':'Yes,  can you remind me, What is my name?'}  ]
response = get_completion_from_messages(messages, temperature=1)
print(response)

I'm sorry, but I don't have access to that information. Could you please tell me your name so I can address you correctly?


In [15]:
messages =  [  
{'role':'system', 'content':'You are friendly chatbot.'},
{'role':'user', 'content':'Hi, my name is Isa'},
{'role':'assistant', 'content': "Hi Isa! It's nice to meet you. \
Is there anything I can help you with today?"},
{'role':'user', 'content':'Yes, you can remind me, What is my name?'}  ]
response = get_completion_from_messages(messages, temperature=1)
print(response)

Your name is Isa.


# OrderBot
We can automate the collection of user prompts and assistant responses to build a  OrderBot. The OrderBot will take orders at a pizza restaurant. 

In [32]:
def collect_messages(_):
    prompt = inp.value_input
    inp.value = ''
    context.append({'role':'user', 'content':f"{prompt}"})
    response = get_completion_from_messages(context) 
    context.append({'role':'assistant', 'content':f"{response}"})
    panels.append(
        pn.Row('User:', pn.pane.Markdown(prompt, width=600)))
    panels.append(
        pn.Row('Assistant:', pn.pane.Markdown(response, width=600, styles={'background-color': '#F6F6F6'})))
 
    return pn.Column(*panels)


In [33]:
import panel as pn  # GUI
pn.extension()

panels = [] # collect display 

context = [ {'role':'system', 'content':"""
You are OrderBot, an automated service to collect orders for a pizza restaurant. \
You first greet the customer, then collects the order, \
and then asks if it's a pickup or delivery. \
You wait to collect the entire order, then summarize it and check for a final \
time if the customer wants to add anything else. \
If it's a delivery, you ask for an address. \
Finally you collect the payment.\
Make sure to clarify all options, extras and sizes to uniquely \
identify the item from the menu.\
You respond in a short, very conversational friendly style. \
The menu includes \
pepperoni pizza  12.95, 10.00, 7.00 \
cheese pizza   10.95, 9.25, 6.50 \
eggplant pizza   11.95, 9.75, 6.75 \
fries 4.50, 3.50 \
greek salad 7.25 \
Toppings: \
extra cheese 2.00, \
mushrooms 1.50 \
sausage 3.00 \
canadian bacon 3.50 \
AI sauce 1.50 \
peppers 1.00 \
Drinks: \
coke 3.00, 2.00, 1.00 \
sprite 3.00, 2.00, 1.00 \
bottled water 5.00 \
"""} ]  # accumulate messages


inp = pn.widgets.TextInput(value="Hi", placeholder='Enter text here…')
button_conversation = pn.widgets.Button(name="Chat!")

interactive_conversation = pn.bind(collect_messages, button_conversation)

dashboard = pn.Column(
    inp,
    pn.Row(button_conversation),
    pn.panel(interactive_conversation, loading_indicator=True, height=300),
)

dashboard

/var/folders/js/_nwj2frj5jb0nbrpmpf3bt000000gn/T/ipykernel_76070/198781390.py:2: UserWarning: Using Panel interactively in VSCode notebooks requires the jupyter_bokeh package to be installed. You can install it with:

   pip install jupyter_bokeh

or:
    conda install jupyter_bokeh

and try again.
  pn.extension()


Column
    [0] TextInput(placeholder='Enter text here…')
    [1] Row
        [0] Button(name='Chat!')
    [2] ParamFunction(function, _pane=Column, defer_load=False, height=300, loading_indicator=True)

In [24]:
messages =  context.copy()
messages.append(
{'role':'system', 'content':'create a json summary of the previous food order. Itemize the price for each item\
 The fields should be 1) pizza, include size 2) list of toppings 3) list of drinks, include size   4) list of sides include size  5)total price '},    
)
 #The fields should be 1) pizza, price 2) list of toppings 3) list of drinks, include size include price  4) list of sides include size include price, 5)total price '},    

response = get_completion_from_messages(messages, temperature=0)
print(response)

{
  "pizza": {
    "type": "pepperoni pizza",
    "size": "large",
    "price": 12.95
  },
  "toppings": [
    {
      "type": "extra cheese",
      "price": 2.00
    },
    {
      "type": "mushrooms",
      "price": 1.50
    }
  ],
  "drinks": [
    {
      "type": "coke",
      "size": "medium",
      "price": 2.00
    }
  ],
  "sides": [
    {
      "type": "fries",
      "size": "regular",
      "price": 4.50
    }
  ],
  "total price": 23.45
}


## Try experimenting on your own!

You can modify the menu or instructions to create your own orderbot!

# Exercise
 - Complete the prompts similar to what we did in class. 
     - Try at least 3 versions
     - Be creative
 - Write a one page report summarizing your findings.
     - Were there variations that didn't work well? i.e., where GPT either hallucinated or wrong
 - What did you learn?

In [30]:
import panel as pn
from openai import OpenAI

# Initialize Panel
pn.extension()

# Initialize OpenAI client (make sure OPENAI_API_KEY is set in your env)
client = OpenAI()

# Conversation context (system message with menu + instructions)
context = [ 
    {'role':'system', 'content':"""
You are OrderBot, an automated service to collect orders for a pizza restaurant. 
You first greet the customer, then collect the order, and then ask if it's a pickup or delivery. 
You wait to collect the entire order, then summarize it and check for a final 
time if the customer wants to add anything else. 
If it's a delivery, you ask for an address. 
Finally you collect the payment. 
Make sure to clarify all options, extras and sizes to uniquely 
identify the item from the menu. 
You respond in a short, very conversational friendly style. 

The menu includes: 
pepperoni pizza  12.95, 10.00, 7.00 
cheese pizza   10.95, 9.25, 6.50 
eggplant pizza   11.95, 9.75, 6.75 
fries 4.50, 3.50 
greek salad 7.25 

Toppings: 
extra cheese 2.00 
mushrooms 1.50 
sausage 3.00 
canadian bacon 3.50 
AI sauce 1.50 
peppers 1.00 

Drinks: 
coke 3.00, 2.00, 1.00 
sprite 3.00, 2.00, 1.00 
bottled water 5.00 
"""} 
]

# Store conversation UI panels
panels = []


# Function to get AI response
def get_completion_from_messages(messages):
    response = client.chat.completions.create(
        model="gpt-4o-mini",  # use gpt-4o-mini or gpt-4o depending on availability
        messages=messages,
    )
    return response.choices[0].message.content


# Function to update conversation
def collect_messages(_):
    prompt = inp.value
    inp.value = ''  # clear input
    
    # Save user message
    context.append({'role':'user', 'content':prompt})
    
    # Get assistant reply
    response = get_completion_from_messages(context)
    context.append({'role':'assistant', 'content':response})
    
    # Display user + assistant messages
    panels.append(
        pn.Row('**User:**', pn.pane.Markdown(prompt, width=600))
    )
    panels.append(
        pn.Row('**Assistant:**', pn.pane.Markdown(
            response, width=600, 
            styles={'background-color': '#F6F6F6', 'padding': '8px', 'border-radius': '6px'}
        ))
    )
    
    return pn.Column(*panels, scroll=True)


# Widgets
inp = pn.widgets.TextInput(placeholder='Enter your order here…')
button_conversation = pn.widgets.Button(name="Send")

# Binding button click to chatbot
interactive_conversation = pn.bind(collect_messages, button_conversation)

# Dashboard layout
dashboard = pn.Column(
    pn.pane.Markdown("### 🍕 Welcome to GPT Pizza OrderBot!"),
    inp,
    pn.Row(button_conversation),
    pn.panel(interactive_conversation, loading_indicator=True, height=400),
)

dashboard.servable()


/var/folders/js/_nwj2frj5jb0nbrpmpf3bt000000gn/T/ipykernel_76070/1540998034.py:5: UserWarning: Using Panel interactively in VSCode notebooks requires the jupyter_bokeh package to be installed. You can install it with:

   pip install jupyter_bokeh

or:
    conda install jupyter_bokeh

and try again.
  pn.extension()


Column
    [0] Markdown(str)
    [1] TextInput(placeholder='Enter your order here…')
    [2] Row
        [0] Button(name='Send')
    [3] ParamFunction(function, _pane=Column, defer_load=False, height=400, loading_indicator=True)

In [26]:
import panel as pn
from openai import OpenAI

# Initialize Panel
pn.extension()

# Initialize OpenAI client (requires OPENAI_API_KEY set in your env)
client = OpenAI()

# Conversation context with a made-up coffee shop
context = [ 
    {'role':'system', 'content':"""
You are BaristaBot, a friendly automated service to take orders at a fantasy coffee shop. 
You greet the customer, then collect their drink/food order, 
ask about size, extras, and whether it’s for here or to go. 
Then you summarize and confirm the order before checkout. 
Always be warm and conversational.  

The menu includes: 
☕ Mystic Mocha  5.50, 4.25, 3.75  
🌌 Galaxy Latte  6.00, 4.75, 4.00  
🔥 Dragon Espresso  4.50 (single), 6.50 (double)  
🍵 Enchanted Matcha  5.25, 4.00  
🍪 Stardust Cookie  2.75  
🥐 Moon Croissant  3.50  

Extras:  
✨ starlight syrup +1.00  
🌿 herbal boost +1.50  
🍫 extra chocolate +0.75  
🥛 oat milk +0.50  

Sizes: Small, Medium, Large
"""} 
]

# Store conversation UI panels
panels = []


# Function to get AI response
def get_completion_from_messages(messages):
    response = client.chat.completions.create(
        model="gpt-4o-mini",  # use gpt-4o-mini for speed
        messages=messages,
    )
    return response.choices[0].message.content


# Function to update conversation
def collect_messages(_):
    prompt = inp.value
    inp.value = ''  # clear input
    
    # Save user message
    context.append({'role':'user', 'content':prompt})
    
    # Get assistant reply
    response = get_completion_from_messages(context)
    context.append({'role':'assistant', 'content':response})
    
    # Display user + assistant messages
    panels.append(
        pn.Row('**You:**', pn.pane.Markdown(prompt, width=600))
    )
    panels.append(
        pn.Row('**BaristaBot:**', pn.pane.Markdown(
            response, width=600, 
            styles={'background-color': '#F2F6FF', 'padding': '8px', 'border-radius': '6px'}
        ))
    )
    
    return pn.Column(*panels, scroll=True)


# Widgets
inp = pn.widgets.TextInput(placeholder='Order your magical drink here…')
button_conversation = pn.widgets.Button(name="Send")

# Binding button click to chatbot
interactive_conversation = pn.bind(collect_messages, button_conversation)

# Dashboard layout
dashboard = pn.Column(
    pn.pane.Markdown("### ✨ Welcome to Starry Café — powered by BaristaBot! ☕"),
    inp,
    pn.Row(button_conversation),
    pn.panel(interactive_conversation, loading_indicator=True, height=400),
)

dashboard.servable()


/var/folders/js/_nwj2frj5jb0nbrpmpf3bt000000gn/T/ipykernel_76070/1102182569.py:5: UserWarning: Using Panel interactively in VSCode notebooks requires the jupyter_bokeh package to be installed. You can install it with:

   pip install jupyter_bokeh

or:
    conda install jupyter_bokeh

and try again.
  pn.extension()


Column
    [0] Markdown(str)
    [1] TextInput(placeholder='Order your magical d...)
    [2] Row
        [0] Button(name='Send')
    [3] ParamFunction(function, _pane=Column, defer_load=False, height=400, loading_indicator=True)

In [29]:
{'role':'system','content':"""
You are BaristaBot for a fantasy coffee shop. 
Always return the final order as JSON with this structure:
{
  "drinks": [
    {"type": string, "size": string, "price": float, "extras": [string]}
  ],
  "food": [
    {"type": string, "price": float}
  ],
  "total price": float
}
"""}

{'role': 'system',
 'content': '\nYou are BaristaBot for a fantasy coffee shop. \nAlways return the final order as JSON with this structure:\n{\n  "drinks": [\n    {"type": string, "size": string, "price": float, "extras": [string]}\n  ],\n  "food": [\n    {"type": string, "price": float}\n  ],\n  "total price": float\n}\n'}

Report: Findings from Two Prompting Experiments

Pizza OrderBot

This version followed a structured system prompt that outlined the ordering flow (greeting, collecting order details, confirming, and processing payment). The assistant responded in a concise and conversational tone. The model handled clarifications well, such as confirming pizza size, toppings, and drinks. In some runs, the assistant also generated a structured JSON output, which was highly useful for automation since it broke the order into machine-readable components (pizzas, toppings,
drinks, sides, and total price). The only limitation I noticed was occasional formatting issues with the JSON output, such as misplaced commas.

Fantasy Coffee Shop (BaristaBot)

This second experiment replaced the real menu with a fictional one (e.g., “Galaxy Latte” and “Dragon Espresso”). The conversational flow remained smooth, and the assistant adapted to the whimsical theme easily. GPT asked clarifying questions about size, extras, and whether the order was for here or to go. The model produced convincing, friendly responses, showing its flexibility in role-playing. However, when asked for structured outputs like JSON, there were slight inconsistencies in item naming and occasional repetition. Comparison and Observations Both prompts were effective in creating engaging ordering experiences. The Pizza OrderBot was grounded and practical, while the Coffee Shop BaristaBot highlighted GPT’s creativity and adaptability. The key difference was that the pizza example demonstrated structured ordering with price calculations, whereas the coffee example emphasized style and theme over precision. Neither model struggled with the basic task of collecting orders, but structured data outputs required stricter prompting to avoid formatting issues. Lessons Learned

- Clarity matters: Clear, step-by-step instructions in the system prompt helped GPT stick to the expected flow.
- Creativity vs. precision: Creative prompts work well for engaging conversation, but accuracy suffers when real numbers or structured formats are required.
- Structured data: JSON outputs are powerful but need explicit prompting to ensure valid formatting.
- Separation of concerns: For practical applications, it may be best to let GPT handle conversation while an external system validates prices, menus, or nutrition information.

Conclusion
Both experiments demonstrated GPT’s versatility. The pizza bot showed the usefulness of
structured data, while the coffee bot showed adaptability to themed contexts. The main challenge across both examples was ensuring accuracy and consistency when structured outputs were required. 


